In [1]:
import torchvision
import torch
from PIL import Image
from sklearn.metrics import confusion_matrix, accuracy_score
import torch.nn as nn
from torch import optim
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import time
import random
from imblearn.under_sampling import RandomUnderSampler

In [2]:
# Replace last classifier to only handle 2 cases, one benign one high grade
num_classes = 2
model = torchvision.models.alexnet(weights = 'AlexNet_Weights.DEFAULT')
# replace the last classifier
model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)

In [3]:
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier[4].parameters():
    param.requires_grad = True
for param in model.classifier[5].parameters():
    param.requires_grad = True
for param in model.classifier[6].parameters():
    param.requires_grad = True

In [4]:
labels = pd.read_csv('case_grade_match.csv')

In [5]:
# Need to group the patches by their cases, and also need to randomly split, the function below will group the cases
def group_patches(patch_dir):
    case_patches = {}
    for filename in os.listdir(patch_dir):
        # If the image size is less than a kilobyte, don't use the patch
        if os.path.getsize(os.path.join(patch_dir, filename)) < 2000:
            continue
        # If 'patched' in filename, not an actual patch
        if 'patched_' in filename:
            continue
        elif filename.endswith('.png'):
            case_num = int(filename.split('_')[1])
            if case_num not in case_patches:
                case_patches[case_num] = []
            case_patches[case_num].append(os.path.join(patch_dir, filename))
    return case_patches

# Generate a Dataset class for the dataloader to feed into cnn model
class PNGDataset(Dataset):
    def __init__(self, case_patches, labels_df, transform=None):
        self.case_patches = case_patches
        self.labels_df = labels_df
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for case_num, patches in case_patches.items():
            label = labels_df.loc[labels_df['Case'] == case_num, 'Class'].values[0]
            label = 0 if label == 1 else 1
            for patch_path in patches:
                self.image_paths.append(patch_path)
                self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        # Get patches, if they are not patch, don't use the patch
        image = Image.open(image_path).convert('RGB')
        # Get the label information using the labels dataframe based on case number
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)
        return image, label

In [6]:
# Group the patches
patches = group_patches('Patches/')

# Get case numbers and their labels
case_nums = list(patches.keys())
dataset = labels.loc[[(int(x)-1) for x in case_nums]]
# Remove those that are equal to 2
noindex = dataset.Class != 2.0
X = dataset[noindex].Case
y = dataset[noindex].Class
train, test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify = y, random_state=40)

# Create the training patches and the test patches
train_patches = {case_num: patches[int(case_num)] for case_num in train}
test_patches = {case_num: patches[int(case_num)] for case_num in test}

# There is an extreme imbalance between benign and high grade, so this code below is meant to address that imbalance
train_patches_flat = []
train_labels_flat = []
for case_num, patches_list in train_patches.items():
    label = labels.loc[labels['Case'] == case_num, 'Class'].values[0]
    label = 0 if label == 1 else 1
    for patch_path in patches_list:
        train_patches_flat.append(patch_path)
        train_labels_flat.append(label)
undersampler = RandomUnderSampler(random_state = 42)
train_patches_resampled, train_labels_resampled = undersampler.fit_resample(np.array(train_patches_flat).reshape(-1, 1), np.array(train_labels_flat))
# Putting the patches back together into a dictionary that can be taken in by the class I created
train_patches_balanced = {}
for patch_path, label in zip(train_patches_resampled.flatten(), train_labels_resampled):
    case_num = int(patch_path.split('_')[1])
    if case_num not in train_patches_balanced:
        train_patches_balanced[case_num] = []
    train_patches_balanced[case_num].append(patch_path)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Generate the actual datasets for the models
train_dataset = PNGDataset(train_patches_balanced, labels, transform=transform)
test_dataset = PNGDataset(test_patches, labels, transform = transform)
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [7]:
# Verifying that the training dataset is indeed equal
np.unique(np.array(train_dataset.labels), return_counts = True)

(array([0, 1]), array([2177, 2177], dtype=int64))

In [8]:
start = time.time()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Training loop
num_epochs = 1000  # Number of epochs
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

patience = 0
best_loss = 100000

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_dataloader:
        images, labels = images.to(device), labels.to(device)


        optimizer.zero_grad()


        outputs = model(images)
        loss = criterion(outputs, labels)


        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()
    
    epoch_loss = running_loss / len(train_dataloader)
    train_accuracy = 100 * correct_train / total_train

    
    if epoch_loss < best_loss:
        #print('Model Saved!')
        #torch.save(model.state_dict(), f'model_epoch_{epoch + 1}.pth')
        best_loss = epoch_loss
        patience = 0
    else:
        patience += 1
    print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {epoch_loss:.4f}, Patience: {patience}, Best Loss: {best_loss:.4f}, Training Accuracy: {train_accuracy:.2f}%')
    
    if patience >= 50:
        print(f'Early stopping at epoch {epoch + 1}')
        break
end = time.time()
elapsed = end - start
minutes = int(elapsed // 60)
seconds = int(elapsed % 60)
print(f'Time taken: {minutes} minutes, {seconds} seconds')

Epoch [1/1000], Loss: 0.7894, Patience: 0, Best Loss: 0.7894, Training Accuracy: 63.85%
Epoch [2/1000], Loss: 0.4754, Patience: 0, Best Loss: 0.4754, Training Accuracy: 76.32%
Epoch [3/1000], Loss: 0.4087, Patience: 0, Best Loss: 0.4087, Training Accuracy: 80.85%
Epoch [4/1000], Loss: 0.3515, Patience: 0, Best Loss: 0.3515, Training Accuracy: 83.62%
Epoch [5/1000], Loss: 0.3480, Patience: 0, Best Loss: 0.3480, Training Accuracy: 85.92%
Epoch [6/1000], Loss: 0.3137, Patience: 0, Best Loss: 0.3137, Training Accuracy: 85.76%
Epoch [7/1000], Loss: 0.2709, Patience: 0, Best Loss: 0.2709, Training Accuracy: 88.17%
Epoch [8/1000], Loss: 0.2639, Patience: 0, Best Loss: 0.2639, Training Accuracy: 88.29%
Epoch [9/1000], Loss: 0.2314, Patience: 0, Best Loss: 0.2314, Training Accuracy: 90.77%
Epoch [10/1000], Loss: 0.2248, Patience: 0, Best Loss: 0.2248, Training Accuracy: 90.06%
Epoch [11/1000], Loss: 0.1840, Patience: 0, Best Loss: 0.1840, Training Accuracy: 92.12%
Epoch [12/1000], Loss: 0.1869,

In [9]:
pred = []
labels = []
with torch.no_grad():
    for images, label in test_dataloader:
        images, label = images.to(device), label.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        labels.append(label)
        pred.append(predicted)

pred = torch.cat(pred).cpu()
labels = torch.cat(labels).cpu()
accuracy = accuracy_score(labels, pred)
print(f'Accuracy: {accuracy}')
confusion_matrix(labels, pred)

Accuracy: 0.6171698963497071


array([[ 829,  798],
       [ 901, 1910]], dtype=int64)